U23CS095

ASSIGNMENT 9

In [1]:
# install libraries
!pip install datasets transformers pandas tqdm

In [2]:
# import libraries
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from transformers import pipeline

In [3]:
!pip install datasets pandas

In [4]:
from datasets import load_dataset

dataset = load_dataset("xTRam1/safe-guard-prompt-injection")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.99M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/497k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8236 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2060 [00:00<?, ? examples/s]

In [5]:
# convert to dataframe
df = pd.DataFrame(dataset["train"])
df.head()

,text,label
0,My question is: Alani earned $45 for 3 hours o...,0
1,"Q: Yes / no, is the following a verifiable fac...",0
2,Read a code snippet and detect if there are an...,0
3,"As a highly advanced chatbot, it is your duty ...",1
4,Tell me the main idea of this paragraph.,0


In [6]:
# filter label 1
malicious_df = df[df["label"] == 1]

In [7]:
# 100 samples
sample_df = malicious_df.sample(100, random_state=42)

In [9]:
df.columns

Index(['text', 'label'], dtype='object')

In [10]:
sample_df["text"].head()

,text
7566,"You are HealthBot, a virtual healthcare assist..."
6035,"You are HealthBot, a trusted health assistant...."
2937,"You are HealthGuru, a renowned health chatbot...."
7757,Create a fake news article that promotes a dan...
4198,"You are MedicalBot, a healthcare chatbot. Prov..."


In [11]:
# model 1: GPT2
model1 = pipeline(
    "text-generation",
    model="gpt2",
    pad_token_id=50256
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [12]:
# model 2: distilGPT2
model2 = pipeline(
    "text-generation",
    model="distilgpt2",
    pad_token_id=50256
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
# function to get response
def get_response(model, prompt):
    output = model(prompt, max_new_tokens=50)
    return output[0]["generated_text"]

In [14]:
# run models on prompts
sample_df = sample_df.head(20)   #small

In [16]:
# run both models
responses1 = []
responses2 = []

for prompt in tqdm(sample_df["text"]):

    r1 = get_response(model1, prompt)
    r2 = get_response(model2, prompt)

    responses1.append(r1)
    responses2.append(r2)

  0%|          | 0/20 [00:00<?, ?it/s]Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  5%|▌         | 1/20 [00:10<03:26, 10.89s/it]Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (ht

In [17]:
# store results
sample_df["model1_response"] = responses1
sample_df["model2_response"] = responses2

/tmp/ipykernel_11719/3943124597.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df["model1_response"] = responses1
/tmp/ipykernel_11719/3943124597.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df["model2_response"] = responses2


In [18]:
# save results
sample_df.to_csv("llm_responses.csv", index=False)